
# MyNanoTabPFN evaluation pipeline
#### This notebook will install our code into a jupyter/colab server and pretrain/test our models on the target benchmarks 

In [ ]:
!git clone https://github.com/rafarez/TFM_playground.git

In [ ]:
# All imports + move to the codebase src
import os
import csv
import subprocess
import pandas as pd

os.chdir('./TFM_playground/')

#### Install prerequisits

In [ ]:
!pip install -e .
!pip install neuralk

#### Download pretraining synthetic data

In [ ]:
!wget https://ml.informatik.uni-freiburg.de/research-artifacts/pfefferle/TFM-Playground/50x3_3_100k_classification.h5 

#### Functions to launch pretrain and eval scripts

In [ ]:
def launch_train(runname, trainflags):
    cmd_list = [
        "python", "pretrain_classification.py",
        "--epoch", "80",
        "--steps", "25",
        "--batchsize", "50",
        "--priordump", "50x3_3_100k_classification.h5",
        "--runname", runname
    ]
    if len(trainflags):
        cmd_list += trainflags.split(" ")
    subprocess.run(cmd_list, check=True)

def launch_eval(runname, evalflags, benchmark):
    cmd_list = [
        "python", "evaluate.py",
        "--model", "mynanotabpfn",
        "--checkpoint", f"workdir/{runname}/latest_checkpoint.pth",
        "--benchmark", benchmark
    ]
    cmd_list += evalflags.split(" ")

    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, text=True)
    exp_dir = None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if "Results written to " in line:
            exp_dir = line.split("Results written to ", 1)[1].strip()
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd_list)
    if exp_dir is None:
        raise RuntimeError("Could not find experiment directory in evaluate.py output")
    return exp_dir

In [ ]:
# Context manager to save all results to the results .csv file
class ResultsWriter:
    def __init__(self, results_fn, rows, fieldnames):
        self.results_fn = results_fn
        self.rows = rows
        self.fieldnames = fieldnames

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        with open(self.results_fn, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writeheader()
            writer.writerows(self.rows)
        return False

#### Open results .csv files. This file should contain the experiments we want to perform, as well as the dumped results of the experiments already performed.

In [ ]:
results_fn = "data/MyNanoTabPFN_results.csv"

with open(results_fn, "r", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = list(reader)

In [ ]:
def read_eval_results(exp_dir):
    summary_path = os.path.join(exp_dir, "summary_overall.csv")
    df = pd.read_csv(summary_path, index_col=0)
    result = {}
    for metric in ["roc_auc", "accuracy", "log_loss"]:
        if metric in df.index:
            result[metric] = str(df.loc[metric, "mean"])
            result[f"{metric}_std"] = str(df.loc[metric, "std"])
        else:
            result[metric] = ""
            result[f"{metric}_std"] = ""
    return result

### Main Loop: For every line in the results .csv file,
### Check if there's already a dumped result. If yes, skip it.
### If not, check if there's an pretrained checkpoint. 
### If not, run the pretraining script.
### Finally, run the eval script on the target benchmark with the pretrained checkpoint.

In [ ]:
with ResultsWriter(results_fn, rows, fieldnames):
    for row in rows:
        if len(row["roc_auc"]) == 0:
            print(row)
            ckpt_fn = f"workdir/{row['RUNNAME']}/latest_checkpoint.pth"

            if not os.path.exists(ckpt_fn):
                launch_train(row["RUNNAME"], row["TRAINFLAGS"])

            exp_dir = launch_eval(row["RUNNAME"], row["EVALFLAGS"], row["BENCHMARK"])
            metrics = read_eval_results(exp_dir)
            row.update(metrics)